In [120]:
import pandas as pd
import numpy as np
sncf_data = pd.read_csv("project_dataset.csv", sep=";")

np.random.seed(0)
sncf_data.head()
sncf_data.shape
sncf_data.info()
sncf_data.describe()
sncf_data.duplicated().sum()


<class 'pandas.DataFrame'>
RangeIndex: 12070 entries, 0 to 12069
Data columns (total 26 columns):
 #   Column                                                                                       Non-Null Count  Dtype
---  ------                                                                                       --------------  -----
 0   Date                                                                                         12010 non-null  str  
 1   Service                                                                                      11830 non-null  str  
 2   Gare de dÃ©part                                                                              12011 non-null  str  
 3   Gare d'arrivÃ©e                                                                              12011 non-null  str  
 4   DurÃ©e moyenne du trajet                                                                     11830 non-null  str  
 5   Nombre de circulations prÃ©vues                          

np.int64(174)

# Creation Filtre pour DATASET CLEAN 

In [62]:
import re 
#fix noms de colonnes (accents)

def fixaccent(s):
    try:
        return s.encode("latin1").decode("utf-8")
    except Exception:
        return s
sncf_data.columns = [fixaccent(c) for c in sncf_data.columns]
#drop les colonnes commentaire 
comment_cols = [c for c in sncf_data.columns if "Commentaire" in c]
sncf_data = sncf_data.drop(columns=comment_cols, errors="ignore")

#nettoyage colonnes texte 
text_cols = ["Service", "Gare de départ", "Gare d'arrivée"]
for c in text_cols:
    if c in sncf_data.columns:
        sncf_data[c] = sncf_data[c].astype("string").str.strip()

# standardisation gares (pour évite "Paris Montparnasse" vs "PARIS MONTPARNASSE")
for c in ["Gare de départ", "Gare d'arrivée"]:
    if c in sncf_data.columns:
        sncf_data[c] = sncf_data[c].str.upper().str.replace(r"\s+", " ", regex=True)

# 3) Nettoyage Date (formats mixtes: 2018-01, 01-2018, 2018/01, espaces)
def parse_month(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip().str.replace("/", "-", regex=False)
    m = x.str.match(r"^\d{2}-\d{4}$", na=False)
    x.loc[m] = x.loc[m].str.split("-", expand=True).apply(lambda r: f"{r[1]}-{r[0]}", axis=1)
    return pd.to_datetime(x, format="%Y-%m", errors="coerce")

sncf_data["Date"] = parse_month(sncf_data["Date"])
sncf_data["year"] = sncf_data["Date"].dt.year
sncf_data["month"] = sncf_data["Date"].dt.month


# Chek doublon 

In [121]:
duplicata_test= sncf_data.duplicated().sum()
print("Doublons :", duplicata_test)
#donc
sncf_data = sncf_data.drop_duplicates(keep="first").copy()
print("Nouvelles dimensions:", sncf_data.shape)

Doublons : 174
Nouvelles dimensions: (11896, 26)


# Clean les donnée numérique 

In [63]:
NA_LIKE = {"NC", "Non communiqué", "NON COMMUNIQUE", "-", ""}

def to_float(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.replace(list(NA_LIKE), np.nan)

    # enlever "min", gérer virgules, espaces
    s = (s.str.lower()
           .str.replace("min", "", regex=False)
           .str.replace(",", ".", regex=False)
           .str.replace(r"\s+", "", regex=True))

    # extraire le nombre (safe)
    s = s.str.extract(r"(-?\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(s, errors="coerce")

# Colonnes à convertir = tout sauf texte/date
exclude = {"Date", "Service", "Gare de départ", "Gare d'arrivée"}
num_cols = [c for c in sncf_data.columns if c not in exclude and c not in ("year", "month")]

for c in num_cols:
    sncf_data[c] = to_float(sncf_data[c])

# Missing Value

In [64]:
#Avant de commencer on fait un gros clean des donnes Nul donv NaN  
#On regarde ce qui manque par colonne pour se faire une idée , en % aussi 
missing = sncf_data.isna().sum().sort_values(ascending=False)
total_missing = missing.sum()

report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct_col": sncf_data.isna().mean() * 100,
})

display(report.head(15))
#On remarquera que les commentaire on beaucoup de missing value ce qui est
#  logique , puis en regardant ce que contient les commentaire on se rend 
# compte qu'il y a pas d'interet a garder ces data

,missing_count,missing_pct_col
Date,155,1.284176
Durée moyenne du trajet,240,1.988401
Gare d'arrivée,59,0.488815
Gare de départ,59,0.488815
Nombre de circulations prévues,240,1.988401
Nombre de trains annulés,239,1.980116
Nombre de trains en retard au départ,240,1.988401
Nombre de trains en retard Ã l'arrivÃ©e,241,1.996686
Nombre trains en retard > 15min,242,2.004971
Nombre trains en retard > 30min,238,1.971831


## Why are they Missing ? Any correlation?


In [65]:
#avant on a remarqué qu'il y a chaque fois a peu pres 240 valeur nul donc pour êtres sure de ne pas passer a cote de qlq chose qu on pourrai remplacer au lieu de supprimer betement
cols = sncf_data.columns.difference(exclude).tolist()

mask = sncf_data[cols].isna()
# Combien de lignes ont au moins 1 NA dans ces colonnes?
print("Row with any NA:", mask.any(axis=1).sum())
# Combien de lignes ont TOUS ces champs en NA?
print("ROws with ALL NA:", mask.all(axis=1).sum())
# Ya combien de NA par ligne sur ces colones dcp
print(mask.sum(axis=1).value_counts().head(10))

Row with any NA: 3954
ROws with ALL NA: 0
0    8116
1    3157
2     682
3     104
4      11
Name: count, dtype: int64


# Test Logique sncf


In [66]:
#tchek si c'est pas tout simplement liée car de base le nombre de train = 0,(donc tout ce qui peut etres lié a la racine du retard)
mean_cols = [c for c in sncf_data.columns if "Retard moyen" in c]
count_cols = [c for c in sncf_data.columns if "Nombre" in c and "retard" in c.lower()]

print("mean_cols:", len(mean_cols))
print("count_cols:", len(count_cols))

mean_cols: 5
count_cols: 5


# Resultat trop petit , on ne prendra pas en compte ce test

In [67]:


df = sncf_data.copy()

#  On ignore les colonnes "Commentaire" (quasi vides / inutiles)
comment_cols = [c for c in df.columns if "Commentaire" in c]
df2 = df.drop(columns=comment_cols, errors="ignore")

n_rows = len(df2)

#  Impact d'un dropna global (sur le dataset utile)
after_drop_all = df2.dropna()
pct_kept_all = 100 * len(after_drop_all) / n_rows

# Distribution des NaN par ligne (dataset utile)
na_per_row = df2.isna().sum(axis=1)
max_na = int(na_per_row.max())
pct_rows_any_na = 100 * (na_per_row.gt(0).mean())

# Colonnes les plus touchées (hors commentaires)
missing_summary = (
    pd.DataFrame({
        "missing_count": df2.isna().sum(),
        "missing_pct": df2.isna().mean() * 100
    })
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

print("Missingness report (hors colonnes Commentaire) ")
print(f"Rows total: {n_rows}")
print(f"Rows with any NA: {na_per_row.gt(0).sum()} ({pct_rows_any_na:.1f}%)")
print(f"Max NA per row: {max_na}")
print("\nTop NA-per-row distribution:")
print(na_per_row.value_counts().head(10))

print("\n=== Why we do NOT use dropna() globally ===")
print(f"Rows kept after df.dropna(): {len(after_drop_all)} / {n_rows} ({pct_kept_all:.2f}%)")
print(" Global dropna() would discard too much data, while missing values are sparse (max 4 per row).")
print("\nTop columns with missing values:")
print(missing_summary.head(15))

Missingness report (hors colonnes Commentaire) 
Rows total: 12070
Rows with any NA: 4198 (34.8%)
Max NA per row: 5

Top NA-per-row distribution:
0    7872
1    3319
2     640
3     177
4      50
5      12
Name: count, dtype: int64

=== Why we do NOT use dropna() globally ===
Rows kept after df.dropna(): 7872 / 12070 (65.22%)
 Global dropna() would discard too much data, while missing values are sparse (max 4 per row).

Top columns with missing values:
                                                    missing_count  missing_pct
Prct retard pour cause matériel roulant                       243     2.013256
Prct retard pour cause infrastructure                         243     2.013256
Prct retard pour causes externes                              242     2.004971
Nombre trains en retard > 15min                               242     2.004971
Retard moyen des trains en retard Ã  l'arrivÃ©e               242     2.004971
Prct retard pour cause gestion en gare et réuti...            242     

## Après suppression des colonnes Commentaire, les valeurs manquantes restent éparses (max 4 NaN par ligne) mais touchent ~34% des lignes. Un dropna() global supprimerait une part importante du dataset (≈ un tiers), donc nous privilégions une stratégie de prétraitement + imputation plutôt que la suppression des lignes 

# Test display


In [68]:
display(sncf_data.head(30))
display(sncf_data.tail(30))

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,...,Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",year,month
0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870.0,5.0,289.0,11.247809,3.693179,...,44.0,8.0,36.134454,31.092437,10.92437,15.966387,5.04,0.840336,2018.0,1.0
1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406.0,1.0,213.0,8.479969,4.567119,...,9.0,4.0,20.0,35.0,16.666667,16.666667,8.333333,3.333333,2018.0,1.0
2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226.0,0.0,21.0,6.239683,0.286283,...,6.0,1.0,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111,2018.0,1.0
3,2018-01-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,71.0,7.235211,0.98,...,18.0,<NA>,33.333333,22.222222,16.666667,20.37037,5.555556,1.851852,2018.0,1.0
4,2018-01-01,National,POITIERS,PARIS MONTPARNASSE,94.0,472.0,4.0,224.0,6.784673,3.229701,...,10.0,0.0,15.789474,45.614035,<NA>,15.789474,1.754386,1.754386,2018.0,1.0
5,2018-01-01,National,ST MALO,PARIS MONTPARNASSE,158.0,92.0,0.0,2.0,1.225,0.0625,...,3.0,1.0,22.222222,66.666667,0.0,11.111111,0.0,0.0,2018.0,1.0
6,2018-01-01,National,TOULOUSE MATABIAU,PARIS MONTPARNASSE,257.0,184.0,0.0,<NA>,11.221739,1.193931,...,12.0,3.0,59.090909,22.727273,4.545455,9.090909,4.545455,0.0,2018.0,1.0
7,2018-01-01,National,VANNES,PARIS MONTPARNASSE,156.0,278.0,1.0,168.0,5.120437,3.131047,...,11.0,3.0,24.137931,44.827586,20.689655,3.448,3.448276,3.448276,2018.0,1.0
8,2018-01-01,National,REIMS,PARIS EST,46.0,230.0,1.0,93.0,7.4,3.107569,...,3.0,0.0,17.948718,41.025641,20.512821,5.128205,0.0,15.384615,2018.0,1.0
9,2018-01-01,National,DOUAI,PARIS NORD,77.0,197.0,3.0,39.0,8.032906,1.802921,...,5.0,3.0,36.363636,18.181818,22.727273,18.181818,4.545455,0.0,2018.0,1.0


,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,...,Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",year,month
12040,2021-01-01,National,MARSEILLE ST CHARLES,TOURCOING,301.0,31.0,0.0,9.0,11.485185,3.154839,...,4.0,3.0,28.571429,0.0,0.0,0.0,28.571429,42.857143,2021.0,1.0
12041,2025-04-01,International,PARIS LYON,GENEVE,191.0,249.0,0.0,36.0,12.659722,1.598126,...,8.0,3.0,28.571429,14.285714,7.142857,21.428571,14.285714,14.285714,2025.0,4.0
12042,2018-05-01,National,PARIS LYON,CHAMBERY CHALLES LES EAUX,181.0,186.0,59.0,13.0,25.314103,2.588189,...,8.0,1.0,33.333333,11.111111,11.111111,27.777778,16.666667,0.0,2018.0,5.0
12043,2019-05-01,National,RENNES,PARIS MONTPARNASSE,99.0,619.0,17.0,368.0,13.597373,8.212292,...,5.0,2.0,24.561404,28.070175,17.54386,15.789474,3.508772,10.526316,2019.0,5.0
12044,2018-05-01,National,MACON LOCHE,PARIS LYON,100.0,234.0,42.0,<NA>,19.529356,9.511285,...,13.0,5.0,36.842105,<NA>,26.315789,15.789474,7.894737,7.894737,2018.0,5.0
12045,2022-03-01,National,PARIS MONTPARNASSE,LAVAL,79.0,256.0,0.0,20.0,12.194167,<NA>,...,3.0,2.0,11.111111,38.888889,11.111111,22.222222,0.0,16.67,2022.0,3.0
12046,2020-07-01,National,PARIS EST,METZ,87.0,240.0,8.0,208.0,1.885256,1.669971,...,4.0,0.0,25.0,25.0,25.0,25.0,0.0,0.0,2020.0,7.0
12047,2020-12-01,National,TOULON,PARIS LYON,246.0,178.0,10.0,86.0,4.15155,1.970635,...,11.0,3.0,50.0,25.0,18.75,6.25,0.0,0.0,2020.0,12.0
12048,2023-12-01,National,NANCY,PARIS EST,95.0,304.0,0.0,84.0,5.892857,1.706689,...,3.0,1.0,17.647059,17.647059,26.470588,23.529412,8.823529,5.882353,2023.0,12.0
12049,2018-03-01,<NA>,PERPIGNAN,PARIS LYON,305.0,163.0,1.0,63.0,8.189418,3.21677,...,14.0,7.0,57.142857,14.285714,14.285714,14.285714,0.0,0.0,2018.0,3.0


In [72]:
# Colonnes texte suspectes (avec chiffres / min / virgules)
sus = []
for c in df.select_dtypes(include=["object","string"]).columns:
    s = df[c].astype("string")
    if s.str.contains(r"\bmin\b|,|NC|\d", regex=True, na=False).any():
        sus.append(c)

print("Text cols suspectes:", sus)
for c in sus:
    print("\n---", c, "---")
    print(df[c].dropna().head(10).tolist())

Text cols suspectes: ['Gare de départ', "Gare d'arrivée"]

--- Gare de départ ---
['BORDEAUX ST JEAN', 'LE MANS', 'PARIS MONTPARNASSE', 'PARIS MONTPARNASSE', 'POITIERS', 'ST MALO', 'TOULOUSE MATABIAU', 'VANNES', 'REIMS', 'DOUAI']

--- Gare d'arrivée ---
['PARIS MONTPARNASSE', 'PARIS MONTPARNASSE', 'LA ROCHELLE VILLE', 'NANTES', 'PARIS MONTPARNASSE', 'PARIS MONTPARNASSE', 'PARIS MONTPARNASSE', 'PARIS MONTPARNASSE', 'PARIS EST', 'PARIS NORD']


In [118]:
raw = pd.read_csv("project_dataset.csv", sep=";", usecols=["Date"], dtype=str)
raw_bad = raw[raw["Date"].isna() | ~raw["Date"].str.contains(r"\d", na=False)]
raw["Date"].value_counts(dropna=False).head(150)

Date
2025-09     135
2018-04     134
2025-07     134
2025-08     132
2018-02     131
           ... 
2021/02       2
2021/05       2
2021-08       2
2021 09       2
2021-11       2
Name: count, Length: 150, dtype: int64

In [119]:
clean_path = "dataset_clean.csv"
df.to_csv(clean_path, sep=";", index=False, encoding="utf-8")
print("ecrit: ", clean_path)

ecrit:  dataset_clean.csv
